In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "6"

In [2]:
import time
from datetime import datetime, timedelta

import datasets as hfds
import torch
import pandas as pd
import torchvision.transforms.v2 as v2
from torch.utils import flop_counter
from matplotlib import pyplot as plt
from tqdm import tqdm

import flat_mae.transforms as flat_transforms
import flat_mae.data as flat_data
import flat_mae.models_mae as models_mae

/admin/home/connor/fmri-fm/.venv/lib/python3.11/site-packages/neuromaps/datasets/utils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [3]:
torch.backends.cudnn.benchmark = True

In [4]:
plt.style.use("../clane.mplstyle")
PLOTW = 3.25  # 6.75 two column width, 0.25 pad

In [5]:
def prefix_crop(sample, num_frames: int = 16):
    sample["bold"] = sample["bold"][:num_frames]
    return sample


def expand_mask(sample):
    sample["mask"] = sample["mask"][None, None]
    return sample


def make_transform(space: str = "flat"):
    # hack, have to redefine to exclude center crop bc mni clips is only 16 frames
    # (idk why ig I forgot to make it 24 frames like the others)
    transform = v2.Compose(
        [
            flat_transforms.ToTensor(),
            prefix_crop,
            flat_transforms.Normalize(mode="frame"),
            flat_transforms.Clip(vmax=3.0),
            flat_transforms.get_unmask(space),
            expand_mask,
        ]
    )
    return transform

In [6]:
dataset_root = "s3://medarc/fmri-datasets/eval"
input_spaces = ["schaefer400", "flat", "mni_cortex"]

datasets = {}
transforms = {}

for space in input_spaces:
    transforms[space] = transform = make_transform(space)
    dataset = hfds.load_dataset(
        "arrow",
        data_files=f"{dataset_root}/hcpya-clips.{space}.arrow/test/*.arrow",
        split="train",
    )
    dataset = flat_data.HFDataset(dataset, transforms[space])
    datasets[space] = dataset

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/36 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

In [7]:
device = torch.device("cuda")

In [8]:
models = {}

for space in input_spaces:
    ckpt_path = f"output/input_space/{space}/pretrain/checkpoint-last.pth"
    models[space] = models_mae.MaskedAutoencoderViT.from_checkpoint(ckpt_path).to(device)

In [9]:
flops = {}
for space in input_spaces:
    sample = datasets[space][0]
    bold = sample["bold"][None].to(device)
    mask = sample["mask"][None].to(device)
    model = models[space]
    counter = flop_counter.FlopCounterMode(display=False)
    with counter:
        model(bold, img_mask=mask, mask_ratio=0.9, with_state=False)
    flops[space] = counter.get_total_flops()
    print(f"{space} {flops[space] / 1e9:.0f}G")

schaefer400 90G
flat 93G
mni_cortex 116G


In [10]:
throughput = {}

batch_size = 32
num_workers = 16

loaders = {
    space: torch.utils.data.DataLoader(
        datasets[space],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        drop_last=True,
    )
    for space in input_spaces
}

for space in input_spaces:
    loader = loaders[space]
    model = models[space]

    total_time = 0.0
    total_samples = 0
    for ii, batch in tqdm(enumerate(loader)):
        bold = batch["bold"].to(device)
        mask = batch["mask"].to(device)
        torch.cuda.synchronize()
        tic = time.perf_counter()
        model.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16):
            loss = model(bold, img_mask=mask, mask_ratio=0.9, with_state=False)
        loss.backward()
        torch.cuda.synchronize()
        step_time = time.perf_counter() - tic
        total_time += step_time
        total_samples += len(bold)
        # restart after one batch per worker warmup
        if (ii + 1) == num_workers:
            total_time = 0.0
            total_samples = 0

    throughput[space] = tput = total_samples / total_time
    print(f"{space} {tput:.0f} samples / s")

63it [00:04, 14.40it/s]

schaefer400 557 samples / s



63it [00:13,  4.79it/s]

flat 574 samples / s



63it [00:20,  3.02it/s]

mni_cortex 460 samples / s


In [11]:
load_throughput = {}

for space in input_spaces:
    loader = loaders[space]
    total_time = 0.0
    total_samples = 0
    tic = time.perf_counter()
    for ii, batch in tqdm(enumerate(loader)):
        bold = batch["bold"].to(device)
        mask = batch["mask"].to(device)
        total_samples += len(bold)
        # restart after one batch per worker warmup
        if (ii + 1) == num_workers:
            tic = time.perf_counter()
            total_samples = 0
    total_time = time.perf_counter() - tic

    load_throughput[space] = tput = total_samples / total_time
    print(f"{space} {tput:.0f} samples / s")

63it [00:00, 123.07it/s]


schaefer400 4066 samples / s


63it [00:09,  6.59it/s]

flat 265 samples / s



63it [00:17,  3.50it/s]

mni_cortex 140 samples / s


In [12]:
sample_dims = {
    "schaefer400": 400,
    "flat": 77763,
    "mni_cortex": 132032,
}

# assume fixed disk read bandwidth 400 MB/s which is fairly standard
# aggregate bandwidth across 8x gpus 3.2 GB/s which is about what I get in practice
disk_bandwidth = 400e6

ideal_load_throughput = {}
for space in input_spaces:
    sample_size = 16 * sample_dims[space] * 2
    ideal_load_throughput[space] = load_tput = disk_bandwidth / sample_size
    print(f"{space} {load_tput:.0f} sample / s")

schaefer400 31250 sample / s
flat 161 sample / s
mni_cortex 95 sample / s


In [13]:
%%bash

for space in schaefer400 flat mni_cortex; do
    path="results/epoch_times_${space}.txt"
    [[ -f $path ]] && continue
    grep 'Train.*Total time' "output/input_space/${space}/pretrain/log.txt" | awk '{print $5}' > "$path"
done


In [14]:
train_times = {}
for space in input_spaces:
    path = f"results/epoch_times_{space}.txt"
    total_time = timedelta()
    with open(path) as f:
        for line in f.readlines():
            t = datetime.strptime(line.strip(), "%H:%M:%S")
            total_time += timedelta(hours=t.hour, minutes=t.minute, seconds=t.second)
    train_times[space] = total_time.total_seconds()
    print(f"{space} {train_times[space] / 3600:.1f} hr")

schaefer400 14.3 hr
flat 29.8 hr
mni_cortex 60.4 hr


In [ ]:
num_params = {}
for space in input_spaces:
    num_params[space] = num = sum(p.numel() for p in models[space].parameters())
    print(f"{space} {num / 1e6:.0f} M")

schaefer400 99 M
flat 100 M
mni_cortex 101 M


In [19]:
df = pd.DataFrame(
    {
        "train_time": train_times,
        "num_params": num_params,
        "flops": flops,
        "compute_tput": throughput,
        "load_tput": load_throughput,
    }
)
df["train_time"] = df["train_time"] / 3600
df["num_params"] = df["num_params"] / 1e6
df["flops"] = df["flops"] / 1e9
df = df.reset_index(drop=False, names="model")

df.to_csv("results/input_space_compute.csv", index=False)

In [27]:
df = pd.read_csv("results/input_space_compute.csv")

df["model"] = ["parcel", "flat", "volume"]
df.columns = [
    "input space",
    "train time (hr)",
    "params (M)",
    "FLOP (G)",
    "compute (samples / s)",
    "load (samples / s)",
]
print(df.to_markdown(index=False, floatfmt=".1f"))
print(
    df.to_latex(
        formatters={"train time (hr)": lambda t: f"{t:.1f}"}, float_format="%.0f", index=False
    )
)

with open("results/input_space_compute.tex", "w") as f:
    f.write(
        df.to_latex(
            formatters={"train time (hr)": lambda t: f"{t:.1f}"}, float_format="%.0f", index=False
        )
    )

| input space   |   train time (hr) |   params (M) |   FLOP (G) |   compute (samples / s) |   load (samples / s) |
|:--------------|------------------:|-------------:|-----------:|------------------------:|---------------------:|
| parcel        |              14.3 |         98.6 |       89.5 |                   556.6 |               4066.1 |
| flat          |              29.8 |         99.7 |       92.7 |                   574.0 |                264.7 |
| volume        |              60.4 |        100.8 |      116.3 |                   459.6 |                140.0 |
\begin{tabular}{lrrrrr}
\toprule
input space & train time (hr) & params (M) & FLOP (G) & compute (samples / s) & load (samples / s) \\
\midrule
parcel & 14.3 & 99 & 90 & 557 & 4066 \\
flat & 29.8 & 100 & 93 & 574 & 265 \\
volume & 60.4 & 101 & 116 & 460 & 140 \\
\bottomrule
\end{tabular}



| input space   |   train time (hr) |   params (M) |   FLOP (G) |   compute (samples / s) |   load (samples / s) |
|:--------------|------------------:|-------------:|-----------:|------------------------:|---------------------:|
| parcel        |              14.3 |         98.6 |       89.5 |                   556.6 |               4066.1 |
| flat          |              29.8 |         99.7 |       92.7 |                   574.0 |                264.7 |
| volume        |              60.4 |        100.8 |      116.3 |                   459.6 |                140.0 |